# Motor de Linkage — Notebook de validação (dados reais)

Adaptação do notebook de validação original (`validacao_backend.ipynb`),
que usava 8 BOs **sintéticos**, pra rodar com **BOs reais**:

- `bos_reconstruidos_clean_b.jsonl` — BOs reconstruídos (texto, data,
  município, naturezas, pessoas, documentos, objetos, coordenadas).
- `bos_reconstruidos_gemma4_e4b_preannotate_20260729_001/annotated_documents.jsonl`
  — entidades extraídas por NER real (modelo `gemma4`), no schema de
  34 tipos esperado pelo motor (`PESSOA_NOME`, `OCORRENCIA`, `PLACA`,
  `DOCUMENTO`, ...). Cobre 10.518 dos 11.838 BOs; os demais caem num
  fallback que deriva entidades dos campos estruturados do próprio
  registro (`pessoas`, `documentos`, `objetos`, `naturezas`).

O cruzamento dos dois arquivos no formato esperado por
`MotorDeLinkage.adicionar_bo` (`id`, `texto`, `data_fato`, `municipio`,
`natureza`, `entidades`) está isolado em `dados_reais.py`
(`carregar_bos_reais`), reaproveitado por este notebook.

Escopo deste notebook: partes 1-4 do original (encoder, dados,
instanciar o motor, caso de uso 1) + parte 8 (adicionar um BO novo a
um índice já existente, simulando reinício de serviço). Os casos de
uso 2 e 3, e a limpeza final, ficam fora deste escopo.

Pré-requisitos: `motor_linkage.py` e `dados_reais.py` na mesma pasta
deste notebook, e os dois arquivos de dados citados acima.

In [ ]:
import sys, os

from motor_linkage import MotorDeLinkage
from dados_reais import carregar_bos_reais, construir_geo_lookup
import numpy as np
import joblib

## 1. Encoder — versão de produção (BERTimbau via sentence-transformers)

Requer `pip install sentence-transformers`. Na primeira execução baixa
o modelo (~400MB) -- depois fica em cache local.

In [ ]:
from sentence_transformers import SentenceTransformer

print("Carregando encoder (primeira vez baixa o modelo, pode demorar)...")
encoder = SentenceTransformer("neuralmind/bert-base-portuguese-cased")

Carregando encoder (primeira vez baixa o modelo, pode demorar)...


## 2. Dados reais — amostra de BOs reconstruídos + entidades do NER

Em vez dos 8 BOs sintéticos do notebook original, carregamos uma
**amostra de 300 BOs reais** (os primeiros 300 registros de
`bos_reconstruidos_clean_b.jsonl`), cruzados com as entidades extraídas
pelo NER real via `carregar_bos_reais`.

`geo_lookup` também vem dos dados reais: em vez de coordenadas fixas
por município (como no notebook original), usamos a lat/lon média de
cada município a partir dos próprios registros que já trazem
coordenada (`construir_geo_lookup`) -- não há base externa de
geocodificação disponível neste pacote.

In [ ]:
geo_lookup = construir_geo_lookup()
print(f"{len(geo_lookup)} municípios com coordenada no geo_lookup.")

bos_reais = carregar_bos_reais(limite=300)
print(f"{len(bos_reais)} BOs reais prontos.")

# amostra de como um BO real ficou, já cruzado com o NER
exemplo = bos_reais[1]
print(f"\nExemplo ({exemplo['id']}):")
print(f"  natureza: {exemplo['natureza']}")
print(f"  municipio: {exemplo['municipio']}")
print(f"  data_fato: {exemplo['data_fato']}")
print(f"  entidades ({len(exemplo['entidades'])}): {exemplo['entidades'][:5]}")

230 municípios com coordenada no geo_lookup.
300 BOs reais prontos.

Exemplo (21F2058049776):
  natureza: Conduzir Veiculo Embriagado
  municipio: GOIANA
  data_fato: 12/01/2021
  entidades (27): [{'texto': '12/01/2020', 'tipo': 'DATA'}, {'texto': '16:30H', 'tipo': 'HORA'}, {'texto': 'Veiculo 01', 'tipo': 'OBJETO_VEICULO'}, {'texto': 'Imputado 01', 'tipo': 'PESSOA_MENCAO'}, {'texto': 'Embriaguez', 'tipo': 'OCORRENCIA'}]


## 3. Instanciar o motor e carregar os BOs

Diferente do notebook original (`clf=None`, fallback por média das
features), aqui usamos o classificador **já treinado**
(`modelo_v1_classificador.joblib`), que é o que deve rodar em produção.

In [ ]:
clf = joblib.load("modelo_v1_classificador.joblib")

motor = MotorDeLinkage(encoder, geo_lookup, clf=clf,
                        tau_geo_km=15.0, tau_dias=30.0,
                        limiar_descricao=0.6)

for bo in bos_reais:
    motor.adicionar_bo(bo, entidades=bo["entidades"])

print(f"BOs carregados no índice: {len(motor.ids)}")

BOs carregados no índice: 300


## 4. Caso de uso 1 — "Dado esse BO, quais estão conectados a ele?"

Testamos com um BO real da amostra que tem bastante entidade extraída
pelo NER (`entidades` mais rico -> mais chance de conexão por
identificador exato, além da similaridade semântica).

In [ ]:
bo_consulta = exemplo["id"]
acima_threshold, top_k = motor.bos_conectados(bo_consulta, k=5, threshold=0.6)

print(f"Consulta: {bo_consulta}\n")
print("Acima do threshold:")
for r in acima_threshold:
    print(f"  {r['bo_id']}: score={r['score']:.3f} | motivo: {r['motivo']}")

print("\nTop 5 (independente de threshold):")
for r in top_k:
    print(f"  {r['bo_id']}: score={r['score']:.3f} | motivo: {r['motivo']}")

Consulta: 21F2058049776

Acima do threshold:
  21F2058061842: score=1.000 | motivo: mesmo(a) DOCUMENTO; descrição parecida ('Odor Etilico' ~ 'Odor Etilico')
  21F2058060774: score=1.000 | motivo: mesmo(a) PESSOA_NOME; descrição parecida ('Odor Etilico' ~ 'Odor Etilico')
  21F2058060608: score=0.999 | motivo: mesmo(a) OBJETO_VEICULO; PESSOA_NOME; descrição parecida ('Veiculo 01' ~ 'Veiculo 01')
  21F2058075989: score=0.999 | motivo: mesmo(a) OBJETO_VEICULO; descrição parecida ('Veiculo 01' ~ 'Veiculo 01')
  21F2058066467: score=0.999 | motivo: mesmo(a) PESSOA_NOME; descrição parecida ('Veiculo 01' ~ 'VEICULO')
  21F2058056855: score=0.994 | motivo: mesmo(a) PESSOA_NOME; descrição parecida ('Olhos Avermelhados' ~ 'Olhos Avermelhados')
  21F2058052211: score=0.986 | motivo: mesmo(a) LOCAL_ESTABELECIMENTO; descrição parecida ('Veiculo 01' ~ 'O Veiculo Descrito')
  21F2058063360: score=0.983 | motivo: mesmo(a) LOCAL_ESTABELECIMENTO; descrição parecida ('Sonolento' ~ 'CELULAR')
  21F20580647

## 8. Adicionando um BO novo ao índice já existente

Simula o fluxo real: o índice já está carregado (de `carregar()`), chega
um BO novo, você só chama `adicionar_bo` normalmente.

Pré-requisito técnico desta seção (persistência, parte 7 do notebook
original, fora do escopo pedido aqui): salvar o índice atual e
recarregá-lo numa instância nova, simulando um reinício de serviço.

In [ ]:
# pré-requisito técnico: salvar o índice atual e recarregar (simula reinício de serviço)
motor.salvar("indice_validacao_reais")
print("Salvo em disco:", [f for f in os.listdir(".") if f.startswith("indice_validacao_reais")])

motor_recarregado = MotorDeLinkage.carregar("indice_validacao_reais", encoder, geo_lookup,
                                             tau_geo_km=15.0, tau_dias=30.0)
print(f"BOs recarregados: {len(motor_recarregado.ids)}")

Salvo em disco: ['indice_validacao_reais_classificador.joblib', 'indice_validacao_reais_embeddings.npy', 'indice_validacao_reais_indice.json']
BOs recarregados: 300


In [ ]:
# BO real, reservado FORA da amostra de 300 usada acima -- mesma natureza
# ("Conduzir Veiculo Embriagado") de vários BOs já indexados, mesmo município
# (Recife), pra dar chance real de conexão por similaridade semântica/geo/data.
bo_novo = carregar_bos_reais(apenas_ids=["23E0097000010"])[0]

print(f"BO novo: {bo_novo['id']} | natureza: {bo_novo['natureza']} | "
      f"municipio: {bo_novo['municipio']} | data: {bo_novo['data_fato']}")

motor_recarregado.adicionar_bo(bo_novo, entidades=bo_novo["entidades"])

acima, top_k_novo = motor_recarregado.bos_conectados(bo_novo["id"], k=5, threshold=0.3)
print(f"\n{bo_novo['id']} conecta com (acima do threshold):")
for r in acima:
    print(f"  {r['bo_id']}: score={r['score']:.3f} | motivo: {r['motivo']}")

print("\nTop 5 (independente de threshold):")
for r in top_k_novo:
    print(f"  {r['bo_id']}: score={r['score']:.3f} | motivo: {r['motivo']}")

BO novo: 23E0097000010 | natureza: Conduzir Veiculo Embriagado/Direção De Veículos Sem Habilitação | municipio: RECIFE | data: 1/1/2023

23E0097000010 conecta com (acima do threshold):
  23E0043000035: score=0.967 | motivo: mesmo(a) descrição parecida ('Automovel Fiat  Veiculo' ~ 'Automovel Fiat Uno Fiat Uno Way')
  23E0043000042: score=0.963 | motivo: mesmo(a) descrição parecida ('Automovel Ford Ecosport Veiculo' ~ 'Motocicleta Honda  Veiculo')
  23E0043000062: score=0.934 | motivo: mesmo(a) descrição parecida ('Automovel Nissan  Veiculo' ~ 'Motocicleta Honda Nxr 160 Bros Honda/Nxr 160 Bros')
  23E0043000005: score=0.926 | motivo: similaridade semântica
  23E0044000006: score=0.926 | motivo: similaridade semântica
  23E0043000028: score=0.925 | motivo: similaridade semântica
  23E0043000001: score=0.919 | motivo: similaridade semântica
  23E0045000059: score=0.919 | motivo: mesmo(a) descrição parecida ('Automovel Fiat  Veiculo' ~ 'Balança De Precisão   Prod Eletroportateis/Eletronico'